## K-Fold Cross Validation (k=5, 10 epochs per fold)

In [1]:
from pathlib import Path
import sys
import time

sys.path.insert(0, str(Path.cwd().resolve().parent.parent))

import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import KFold
from ultralytics import YOLO
import torch

import src.config as cfg
from src.training.training_logger import TrainingLogger

print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

NVIDIA GeForce RTX 4060 Ti


In [2]:
ROOT = Path.cwd().resolve().parent.parent
PRETRAINED_DIR = ROOT / "pretrained_models"
OUTPUT_DIR = Path.cwd()

DATASET_DIR = cfg.get_dataset("minecraft_mobs-2")["dir"]

K_FOLDS = 5
EPOCHS = 10
MODEL_SIZE = "s"
BATCH_SIZE = 16
IMGSZ = 768
SEED = 42

logger = TrainingLogger(log_dir=str(ROOT / "training_logs"))

with open(DATASET_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)

NUM_CLASSES = data_cfg["nc"]
CLASS_NAMES = data_cfg["names"]
print(f"{NUM_CLASSES} classes, dataset: {DATASET_DIR}")

16 classes, dataset: C:\Users\pedro\OneDrive\Documentos\programação\eletiva-leomar\YOLOCraft\data\minecraft_mobs-2\apresentacao


In [3]:
image_paths = []
for split in ["train", "val", "test"]:
    images_dir = DATASET_DIR / split / "images"
    if images_dir.exists():
        image_paths += sorted(images_dir.glob("*.jpg"))

image_paths = np.array(image_paths)
print(f"total images: {len(image_paths)}")

total images: 4797


In [4]:
FOLDS_DIR = OUTPUT_DIR / "kfold_splits"
FOLDS_DIR.mkdir(exist_ok=True)

kfold = KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

fold_files = []
for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(image_paths)):
    train_txt = FOLDS_DIR / f"fold{fold_idx}_train.txt"
    val_txt = FOLDS_DIR / f"fold{fold_idx}_val.txt"

    train_txt.write_text("\n".join(str(p) for p in image_paths[train_idx]), encoding="utf-8")
    val_txt.write_text("\n".join(str(p) for p in image_paths[val_idx]), encoding="utf-8")

    names_lines = [f"  {i}: {CLASS_NAMES[i]}" for i in sorted(CLASS_NAMES)]
    yaml_path = FOLDS_DIR / f"fold{fold_idx}_data.yaml"
    yaml_path.write_text(
        "\n".join([f"train: {train_txt}", f"val: {val_txt}", f"nc: {NUM_CLASSES}", "names:"] + names_lines),
        encoding="utf-8",
    )

    fold_files.append(yaml_path)
    print(f"fold {fold_idx}: train={len(train_idx)} val={len(val_idx)}")

fold 0: train=3837 val=960
fold 1: train=3837 val=960
fold 2: train=3838 val=959
fold 3: train=3838 val=959
fold 4: train=3838 val=959


In [ ]:
results_per_fold = []

for fold_idx, yaml_path in enumerate(fold_files):
    print(f"\n=== fold {fold_idx + 1}/{K_FOLDS} ===")

    model = YOLO(str(PRETRAINED_DIR / f"yolo26{MODEL_SIZE}.pt"))

    start = time.time()
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        device=0 if torch.cuda.is_available() else "cpu",
        project=str(OUTPUT_DIR / "runs"),
        name=f"fold{fold_idx}",
    )
    train_time = time.time() - start

    metrics = model.val(data=str(yaml_path), workers=0)

    fold_result = {
        "fold": fold_idx,
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "map50": float(metrics.box.map50),
        "map50_95": float(metrics.box.map),
        "train_time_min": train_time / 60,
    }
    results_per_fold.append(fold_result)

    logger.log_training(
        model_size=MODEL_SIZE,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        imgsz=IMGSZ,
        map50=fold_result["map50"],
        map50_95=fold_result["map50_95"],
        train_time=train_time,
        notes=f"kfold cv fold {fold_idx + 1}/{K_FOLDS}",
    )

    print(f"mAP50={fold_result['map50']:.4f}  mAP50-95={fold_result['map50_95']:.4f}")

In [ ]:
df = pd.DataFrame(results_per_fold)
print(df)

summary = df[["precision", "recall", "map50", "map50_95"]].agg(["mean", "std"])
print("\n=== summary (mean +/- std across folds) ===")
print(summary)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(df["fold"], df["map50"])
plt.axhline(df["map50"].mean(), color="red", linestyle="--", label="mean")
plt.xlabel("fold")
plt.ylabel("mAP50")
plt.title(f"mAP50 per fold (k={K_FOLDS}, epochs={EPOCHS})")
plt.legend()
plt.show()